# SDF-YOLO v2 on GC10-DET (native resolution) (local GPU)

GC10-DET: 10 classes (punching_hole, welding_line, crescent_gap, water_spot, oil_spot, silk_spot, inclusion, rolled_pit, crease, waist_folding), **native 2048×1000** images with per-box labels from the XMLs, split 1602 / 229 / 461 (frozen, `splits/gc10_split_v1.csv`). Repeat-factor sampling (`--rfs 0.1`) oversamples rare classes such as crease and rolled_pit.

This notebook trains **SDF-YOLO-s** and the same-protocol baselines (**YOLO11s**, optionally **YOLOv8s**) on the same data, split and settings, then evaluates each one **once on the TEST split**. The metrics are mAP50, mAP50-95, F1, P, R, GFLOPs, Params and FPS, plus per-class AP50.

Runs on this machine's **NVIDIA RTX 3060**. Task Manager may list it as "GPU 1", but CUDA calls it device **`0`**. Open the notebook from this folder (next to `train.py`) and press **Run All**: training starts directly, same as the Colab commands (`train.py` defaults: 200 epochs, batch 32, imgsz 640, cache ram).

Runs go **one at a time** on the GPU. If another `train.py` (e.g. from the other notebook) is already training, this one prints *waiting in queue* and starts by itself when that run ends. Running two at once on the 12 GB card made each iteration take ~60 s and crashed with `illegal memory access`.

## 1. Setup

In [ ]:
import sys, shutil
from pathlib import Path

DATASET = "gc10"
PY = sys.executable   # train.py runs with the same Python as this kernel
PROJECT_DIR = Path.cwd()
assert (PROJECT_DIR / "train.py").exists(), f"open this notebook from the folder containing train.py (cwd={PROJECT_DIR})"
!"{PY}" -c "import torch, ultralytics; assert torch.cuda.is_available(), 'CUDA not available'; print('torch', torch.__version__, '| ultralytics', ultralytics.__version__, '|', torch.cuda.get_device_name(0))"
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv


In [ ]:
import os, re, codecs, subprocess
from IPython.display import display

def train(args):
    """Run train.py and stream its output live. IPython's `!` on Windows only prints
    once the process has exited, so a 200-epoch run would stay blank for hours
    (Colab runs on Linux, where `!` streams). Progress bars update in place.
    Stopping this cell (Interrupt) also stops the training process."""
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"}
    p = subprocess.Popen([PY, "-u", "train.py", *args.split()], cwd=PROJECT_DIR, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    dec = codecs.getincrementaldecoder("utf-8")(errors="replace")
    buf, bar = "", None   # bar: display handle of the progress line redrawn with carriage returns
    try:
        while chunk := p.stdout.read1(4096):
            buf += dec.decode(chunk)
            *segs, buf = re.split(r"(\r\n|\r|\n)", buf) if buf else [""]
            for seg, sep in zip(segs[::2], segs[1::2]):
                if sep == "\r":                      # progress bar redraw
                    if seg.strip():
                        if bar is None:
                            bar = display({"text/plain": seg}, raw=True, display_id=True)
                        else:
                            bar.update({"text/plain": seg}, raw=True)
                elif bar is not None:                # bar finished: freeze its last state
                    if seg.strip():
                        bar.update({"text/plain": seg}, raw=True)
                    bar = None
                else:
                    print(seg, flush=True)
        if buf.strip():
            print(buf)
        rc = p.wait()
    except KeyboardInterrupt:
        p.kill()
        p.wait()
        print("training stopped")
        raise
    if rc:
        raise RuntimeError(f"train.py exited with code {rc} -- see the output above")


## 2. Dataset check

In [ ]:
EXPECTED = {'train': 1602, 'val': 229, 'test': 461}
root = PROJECT_DIR / "datasets" / DATASET
for split, n in EXPECTED.items():
    imgs = len(list((root / "images" / split).glob("*.jpg")))
    lbls = len(list((root / "labels" / split).glob("*.txt")))
    print(f"{split:<5} images={imgs:<5} labels={lbls:<5} expected={n}")
    assert imgs == n, f"{split}: expected {n} images, found {imgs}"
print("dataset OK")

### GC10 options (if you have spare time or GPU)

Add the flags to the train command:

| Option | Flags | Note |
|---|---|---|
| Bigger input (small defects) | `--imgsz 1024 --batch 16` | about 2.5x slower |
| Oversampling stronger or off | `--rfs 0.2` / `--rfs 0` | choose by **val** mAP, never test |
| RAM crash | `--cache disk` | |
| CUDA out of memory | `--batch 16` | |


## 3. Train SDF-YOLO-s

Always train through `train.py`, never by importing it. That way `CAA` pickles as `__main__.CAA` and `best.pt` stays loadable elsewhere.

In [ ]:
train(f"--data {DATASET} --model sdf")

## 4. Baseline: YOLO11s (same data, split and settings)

In [ ]:
train(f"--data {DATASET} --model yolo11s.pt")


## 5. Baseline: YOLOv8s

In [ ]:
train(f"--data {DATASET} --model yolov8s.pt")


## 6. Baseline: YOLO11n (nano)

In [ ]:
train(f"--data {DATASET} --model yolo11n.pt")


## 7. Baseline: YOLOv5s

In [ ]:
train(f"--data {DATASET} --model yolov5su.pt")


## 8. Baseline: YOLOv9s

In [ ]:
train(f"--data {DATASET} --model yolov9s.pt")


## 9. Baseline: YOLOv10s

In [ ]:
train(f"--data {DATASET} --model yolov10s.pt")


## 10. Baseline: YOLO12s

In [ ]:
train(f"--data {DATASET} --model yolo12s.pt")


GPU: NVIDIA GeForce RTX 3060
torch 2.11.0+cu128 | ultralytics 8.3.174

######################################################################
# gc10_yolo12s_i1024
######################################################################
repeat-factor sampling: {'images': 1602, 'after_rfs': 1723, 'repeat_per_class': {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0, 6: 1.08, 7: 2.31, 8: 2.03, 9: 1.28}}
Ultralytics 8.3.174  Python-3.12.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=20, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\Shivapreetham\Desktop\colab_sdf_yolo_v2\datasets\gc10\gc10_yolo12s_i1024_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, 

train: Scanning C:\Users\Shivapreetham\Desktop\colab_sdf_yolo_v2\datasets\gc10\labels\train.cache... 1723 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1723/1723 [00:00<?, ?it/s]

WARNING cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.


train: Caching images (0.8GB RAM):  34%|███▍      | 588/1723 [00:03<00:07, 157.07it/s]

## 11. Re-evaluate finished runs (adds mAP75 / per-class AP75)

Runs trained before mAP75 was added have no AP75 in their `summary.json`. This re-tests their existing `best.pt` on the TEST split (no retraining, a minute or so each) and skips runs that already have it.

In [ ]:
import json
RUNS = {"sdf": f"{DATASET}_sdf_s", "yolo11s.pt": f"{DATASET}_yolo11s", "yolov8s.pt": f"{DATASET}_yolov8s",
        "yolo11n.pt": f"{DATASET}_yolo11n",
        "yolov5su.pt": f"{DATASET}_yolov5su",
        "yolov9s.pt": f"{DATASET}_yolov9s",
        "yolov10s.pt": f"{DATASET}_yolov10s",
        "yolo12s.pt": f"{DATASET}_yolo12s"}
for model, run in RUNS.items():
    s = Path("runs", run, "summary.json")
    has_ap75 = s.exists() and "mAP75" in json.loads(s.read_text())["test_split"]
    if Path("runs", run, "weights", "best.pt").exists() and not has_ap75:
        train(f"--data {DATASET} --model {model} --eval-only")


## 12. Results (TEST split)

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(Path("runs/results_table.csv"))
df = df[df.dataset == DATASET]
display(df.set_index("run")[["model", "mAP50", "mAP75", "mAP50_95", "F1", "Precision", "Recall",
                             "GFLOPs", "Params_M", "FPS_bs1"]])

summaries = {run: json.loads(Path("runs", run, "summary.json").read_text())["test_split"]
             for run in df.run if Path("runs", run, "summary.json").exists()}
for k in ("per_class_AP50", "per_class_AP75", "per_class_AP50_95"):
    per_class = {run: t[k] for run, t in summaries.items() if k in t}
    if per_class:
        print(f"{k[len('per_class_'):]} per class (test)")
        display(pd.DataFrame(per_class))


In [ ]:
# Training curves and test confusion matrix per run
for run in df.run:
    for f in (Path("runs", run, "results.png"),
              Path("runs", f"{run}_test", "confusion_matrix_normalized.png")):
        if f.exists():
            print(f)
            display(Image(filename=str(f), width=900))

## 13. Zip results

In [ ]:
out = shutil.make_archive(f"sdf_v2_results_gc10", "zip", root_dir=PROJECT_DIR, base_dir="runs")
print("saved", out)